# 1. Data Loading and Interaction Matrix Construction

- start by loading the playlist and track data, then build an interaction matrix where each row represents a playlist and each column represents a unique track. A cell is set to 1 if the track is in the playlist.

In [63]:
import pandas as pd
import numpy as np

# Load playlists and tracks (using a subset for faster experiments)
playlists = pd.read_parquet('playlist_full.parquet')[:500]
tracks = pd.read_parquet('track_full.parquet')

# Extract playlist ids and track lists
playlist_ids = playlists['playlist_idx'].tolist()
playlist_tracks = dict(zip(playlists['playlist_idx'], playlists['track_idx_list']))

# Create a sorted list of all unique track ids
unique_tracks = set()
for tlist in playlist_tracks.values():
    unique_tracks.update(tlist)
unique_tracks = sorted(list(unique_tracks))

# Map track ids to column indices
track_to_col = {track: idx for idx, track in enumerate(unique_tracks)}

# Build the interaction matrix: rows = playlists, columns = tracks
interaction_matrix = np.zeros((len(playlist_ids), len(unique_tracks)))
for i, pid in enumerate(playlist_ids):
    for track in playlist_tracks[pid]:
        j = track_to_col[track]
        interaction_matrix[i, j] = 1  # Use count if necessary

print("Interaction matrix shape:", interaction_matrix.shape)

Interaction matrix shape: (500, 22118)


# 2. Collaborative Filtering Methods

- We implement three different recommendation methods.



### 2.1. Item-Based Collaborative Filtering
- For item-based CF, we compute similarities between tracks based on the playlists in which they appear. For a given track, find similar tracks that tend to co-occur in playlists, then recommend those.

In [64]:
from sklearn.metrics.pairwise import cosine_similarity

def compute_item_similarity(train_matrix):
    """Compute cosine similarity between tracks (columns)."""
    return cosine_similarity(train_matrix.T)

def predict_item_based(playlist_idx, train_matrix, sim_matrix=None):
    """
    Predict scores for a given playlist using item-based CF.
    
    Parameters:
      - playlist_idx: Index of the playlist.
      - train_matrix: The interaction matrix.
      - sim_matrix: Precomputed item similarity matrix (optional).
      
    Returns:
      - 1D array of predicted scores for each track.
    """
    playlist_vector = train_matrix[playlist_idx].reshape(1, -1)
    
    # Use precomputed similarity matrix if available
    if sim_matrix is not None:
        predicted_scores = playlist_vector.dot(sim_matrix)
    else:
        sim = cosine_similarity(train_matrix.T)
        predicted_scores = playlist_vector.dot(sim)
    
    # Remove tracks already in the playlist
    existing_track_indices = np.where(playlist_vector.flatten() > 0)[0]
    predicted_scores[0, existing_track_indices] = -np.inf
    
    return predicted_scores.flatten()

### 2.2. User-Based Collaborative Filtering

- For user-based CF, we compute similarities between playlists. For a given playlist, find other similar playlists and recommend tracks that are present in these similar playlists but missing in the current one.

In [65]:
def compute_user_similarity(train_matrix):
    """Compute cosine similarity between playlists (rows)."""
    return cosine_similarity(train_matrix)

def predict_user_based(playlist_idx, train_matrix, user_sim_matrix=None):
    """
    Predict scores for a given playlist using user-based CF.
    
    Parameters:
      - playlist_idx: Index of the playlist.
      - train_matrix: The interaction matrix.
      - user_sim_matrix: Precomputed user similarity matrix (optional).
      
    Returns:
      - 1D array of predicted scores for each track.
    """
    if user_sim_matrix is None:
        user_sim_matrix = compute_user_similarity(train_matrix)
    
    sim_scores = user_sim_matrix[playlist_idx]
    predicted_scores = sim_scores.dot(train_matrix)
    
    # Remove tracks already present in the playlist
    playlist_vector = train_matrix[playlist_idx]
    existing_track_indices = np.where(playlist_vector.flatten() > 0)[0]
    predicted_scores[existing_track_indices] = -np.inf
    
    return predicted_scores

### 2.3. Matrix Factorization using Truncated SVD
- We use truncated SVD to factorize the interaction matrix into latent factors. Then, recommendations are generated by computing the dot product of the latent factors.

In [66]:
from sklearn.decomposition import TruncatedSVD

# Choose the latent dimension (number of factors)
latent_dim = 100

# Perform truncated SVD
svd = TruncatedSVD(n_components=latent_dim, random_state=42)
U = svd.fit_transform(interaction_matrix)      # Shape: (num_playlists, latent_dim)
Sigma = svd.singular_values_                     # Shape: (latent_dim,)
VT = svd.components_                             # Shape: (latent_dim, num_tracks)

# Rescale using square root of singular values
sqrt_sigma = np.sqrt(Sigma)

# Obtain latent factors P (for playlists) and Q (for tracks)
P = U * sqrt_sigma         # Shape: (num_playlists, latent_dim)
Q = (VT.T * sqrt_sigma)    # Shape: (num_tracks, latent_dim)

print("P shape (playlist factors):", P.shape)
print("Q shape (track factors):", Q.shape)

def predict_mf(playlist_idx, train_matrix, P, Q):
    """
    Predict scores using matrix factorization latent factors.
    
    Parameters:
      - playlist_idx: Index of the playlist.
      - train_matrix: Interaction matrix (used for filtering).
      - P: Playlist latent factor matrix.
      - Q: Track latent factor matrix.
      
    Returns:
      - 1D array of predicted scores for each track.
    """
    predicted_scores = P[playlist_idx].dot(Q.T)
    
    # Remove tracks already in the playlist
    playlist_vector = train_matrix[playlist_idx]
    existing_track_indices = np.where(playlist_vector.flatten() > 0)[0]
    predicted_scores[existing_track_indices] = -np.inf
    
    return predicted_scores

P shape (playlist factors): (500, 100)
Q shape (track factors): (22118, 100)


### 2.4. Matrix Factorization using ALS
- Use ALS on a sparse implicit feedback matrix to learn latent factors for playlists and tracks, then generate recommendations by computing the dot product of these factors.

In [67]:
import numpy as np
from scipy.sparse import csr_matrix
import implicit

# Assume interaction_matrix is already created (dense NumPy array from previous code).
# Convert the dense interaction matrix into a sparse CSR matrix.
sparse_interaction = csr_matrix(interaction_matrix)

# Set parameters for ALS
latent_dim = 100  # You can experiment with different dimensions
regularization = 0.1
iterations = 20
alpha = 40  # Confidence scaling factor for implicit feedback

# Scale the data by confidence: higher weight for observed interactions.
data_conf = (sparse_interaction * alpha).astype('double')

# Initialize and train the ALS model.
als_model = implicit.als.AlternatingLeastSquares(factors=latent_dim,
                                                 regularization=regularization,
                                                 iterations=iterations,
                                                 random_state=42)
# Fit the model using the weighted implicit feedback.
als_model.fit(data_conf)

# Extract latent factors for playlists and tracks.
# Note: In the implicit package, 'user_factors' correspond to the rows of the sparse matrix
# and 'item_factors' correspond to its columns.
P_als = als_model.user_factors   # shape: (num_playlists, latent_dim)
Q_als = als_model.item_factors   # shape: (num_tracks, latent_dim)

print("P_als shape (playlist factors):", P_als.shape)
print("Q_als shape (track factors):", Q_als.shape)

def predict_als(playlist_idx, train_matrix, P_als, Q_als):
    """
    Predict scores using ALS latent factors.
    
    Parameters:
      - playlist_idx: Index of the playlist.
      - train_matrix: The original dense interaction matrix (used for filtering).
      - P_als: Playlist latent factor matrix from ALS.
      - Q_als: Track latent factor matrix from ALS.
      
    Returns:
      - 1D array of predicted scores for each track.
    """
    # Compute predicted scores as the dot product of the playlist's latent factors and all track factors.
    predicted_scores = P_als[playlist_idx].dot(Q_als.T)
    
    # Filter out tracks already in the playlist.
    playlist_vector = train_matrix[playlist_idx]
    existing_track_indices = np.where(playlist_vector.flatten() > 0)[0]
    predicted_scores[existing_track_indices] = -np.inf
    
    return predicted_scores

  0%|          | 0/20 [00:00<?, ?it/s]

P_als shape (playlist factors): (500, 100)
Q_als shape (track factors): (22118, 100)


# 3. Evaluation 
- We evaluate our recommendation models using a leave-one-out strategy on each playlist. We compute the following metrics:
1. Hit Ratio @ K: Fraction of playlists where the held-out track appears in the top K recommendations. A higher value means a greater fraction of playlists have the held-out item in the top K recommendations.
$$
HR@K = \frac{1}{N} \sum_{i=1}^{N} \mathbf{1}{\text{test item in top-}K \text{ for playlist } i}
$$

2. Mean Reciprocal Rank (MRR): The average of the reciprocal ranks of the held-out tracks across playlists. A higher MRR means the relevant item tends to appear closer to the top of the ranked list.
$$
MRR = \frac{1}{N} \sum_{i=1}^{N} \frac{1}{\text{rank}_i}
$$

3. Normalized Discounted Cumulative Gain (NDCG) @ K: This metric accounts for the position of the held-out track in the ranked list. A higher NDCG indicates that the held-out item is not only present but is ranked in a more favorable position relative to an ideal ranking. For a single relevant (held-out) item:
$$
DCG_i = \frac{1}{\log_2(\text{rank}_i + 1)}
$$

- The ideal DCG (IDCG) for a perfect ranking (where the held-out track is at rank 1) is:

$$
IDCG = \frac{1}{\log_2(1 + 1)} = 1
$$

- Then, NDCG for playlist i is:

$$
NDCG_i = \frac{DCG_i}{IDCG} = \frac{1}{\log_2(\text{rank}i + 1)}
$$

- Finally, average over all playlists:

$$
NDCG@K = \frac{1}{N} \sum{i=1}^{N} NDCG_i
$$


### 3.1. Data Splitting & Metrics Functions
- For each playlist:
    - Randomly remove one track and save it as the test item.
	- Use the remaining tracks to construct your interaction matrix.

In [68]:
def leave_one_out_split(interaction_matrix):
    """
    For each playlist (row), randomly hide one track to serve as the test item.
    
    Returns:
      - train_matrix: Interaction matrix with one track removed per playlist.
      - test_items: Dictionary mapping playlist indices to the held-out track index.
    """
    train_matrix = interaction_matrix.copy()
    test_items = {}
    for i in range(train_matrix.shape[0]):
        present_indices = np.where(train_matrix[i] > 0)[0]
        if len(present_indices) == 0:
            continue  # Skip empty playlists
        test_idx = np.random.choice(present_indices)
        train_matrix[i, test_idx] = 0  # Remove test item
        test_items[i] = test_idx
    return train_matrix, test_items

def compute_metrics_for_playlist(predicted_scores, test_idx, k=10):
    """
    Compute Hit, MRR, and NDCG for a single playlist based on predicted scores.
    
    Parameters:
      - predicted_scores: 1D array of scores for all tracks.
      - test_idx: Index of the held-out track.
      - k: Top K recommendations to consider.
      
    Returns:
      - hit: 1 if test_idx is in the top k, else 0.
      - mrr: Reciprocal rank of test_idx.
      - ndcg: Normalized discounted cumulative gain.
    """
    ranked_indices = np.argsort(-predicted_scores)
    hit = 1 if test_idx in ranked_indices[:k] else 0
    
    try:
        rank = np.where(ranked_indices == test_idx)[0][0] + 1  # 1-indexed rank
    except IndexError:
        rank = k + 1  # If not found in ranking
    
    mrr = 1.0 / rank
    ndcg = 1.0 / np.log2(rank + 1) if rank <= k else 0
    return hit, mrr, ndcg

def evaluate_model(train_matrix, test_items, model_predict, k=10):
    """
    Generic evaluation function for recommendation models.
    
    Parameters:
      - train_matrix: Interaction matrix with held-out items removed.
      - test_items: Dictionary mapping playlist index to held-out track index.
      - model_predict: Function that returns predicted scores for a given playlist.
      - k: Number of top recommendations to consider.
      
    Returns:
      - Average Hit Ratio, MRR, and NDCG over all playlists.
    """
    hit_total = 0
    mrr_total = 0
    ndcg_total = 0
    num_playlists = len(test_items)
    
    for playlist_idx, test_idx in test_items.items():
        predicted_scores = model_predict(playlist_idx, train_matrix)
        hit, mrr, ndcg = compute_metrics_for_playlist(predicted_scores, test_idx, k)
        hit_total += hit
        mrr_total += mrr
        ndcg_total += ndcg
    
    hit_ratio = hit_total / num_playlists
    mrr_avg = mrr_total / num_playlists
    ndcg_avg = ndcg_total / num_playlists
    return hit_ratio, mrr_avg, ndcg_avg

# 4. Model Evaluation

- For each method, we create a wrapper that uses precomputed similarity matrices or latent factors, then evaluate using the generic function.

### 4.1. Evaluating Item-Based CF

In [69]:
# For reproducibility
np.random.seed(42)

# Create train-test split
train_matrix, test_items = leave_one_out_split(interaction_matrix)

# Precompute item similarity for efficiency
sim_matrix = compute_item_similarity(train_matrix)

# Define a wrapper for item-based prediction using the precomputed similarity matrix
def predict_item_based_wrapper(playlist_idx, train_matrix):
    return predict_item_based(playlist_idx, train_matrix, sim_matrix)

# Evaluate Item-Based CF
hit_ratio, mrr, ndcg = evaluate_model(train_matrix, test_items, predict_item_based_wrapper, k=10)
print("Item-based CF - Hit Ratio @10:", hit_ratio)
print("Item-based CF - MRR @10:", mrr)
print("Item-based CF - NDCG @10:", ndcg)
print("Scores:", hit_ratio + mrr + ndcg)

Item-based CF - Hit Ratio @10: 0.058
Item-based CF - MRR @10: 0.027676261542049198
Item-based CF - NDCG @10: 0.03083157124574434
Scores: 0.11650783278779354


### 4.2. Evaluating User-Based CF

In [70]:
# For reproducibility
np.random.seed(42)

# Create train-test split
train_matrix, test_items = leave_one_out_split(interaction_matrix)

# Precompute user similarity for efficiency
user_sim_matrix = compute_user_similarity(train_matrix)

# Define a wrapper for user-based prediction using the precomputed similarity matrix
def predict_user_based_wrapper(playlist_idx, train_matrix):
    return predict_user_based(playlist_idx, train_matrix, user_sim_matrix)

# Evaluate User-Based CF
hit_ratio, mrr, ndcg = evaluate_model(train_matrix, test_items, predict_user_based_wrapper, k=10)
print("User-based CF - Hit Ratio @10:", hit_ratio)
print("User-based CF - MRR @10:", mrr)
print("User-based CF - NDCG @10:", ndcg)
print("Scores:", hit_ratio + mrr + ndcg)

User-based CF - Hit Ratio @10: 0.038
User-based CF - MRR @10: 0.02405222816855647
User-based CF - NDCG @10: 0.022417766717207983
Scores: 0.08446999488576445


### 4.3. Evaluating Matrix Factorization

#### 4.3.1 SVD

In [71]:
# For reproducibility
np.random.seed(42)

# (Re)create train-test split if needed
train_matrix, test_items = leave_one_out_split(interaction_matrix)

# Define a wrapper for matrix factorization prediction using latent factors P and Q
def predict_mf_wrapper(playlist_idx, train_matrix):
    return predict_mf(playlist_idx, train_matrix, P, Q)

# Evaluate Matrix Factorization
hit_ratio, mrr, ndcg = evaluate_model(train_matrix, test_items, predict_mf_wrapper, k=10)
print("MF-SVD - Hit Ratio @10:", hit_ratio)
print("MF-SVD - MRR @10:", mrr)
print("MF-SVD - NDCG @10:", ndcg)
print("Scores:", hit_ratio + mrr + ndcg)

MF-SVD - Hit Ratio @10: 0.304
MF-SVD - MRR @10: 0.20589818044346733
MF-SVD - NDCG @10: 0.22181401868306952
Scores: 0.7317121991265368


#### 4.3.2 ALS

In [72]:
# For reproducibility
np.random.seed(42)

# (Re)create train-test split if needed
train_matrix, test_items = leave_one_out_split(interaction_matrix)

# Define a wrapper for ALS prediction using the latent factors P_als and Q_als
def predict_als_wrapper(playlist_idx, train_matrix):
    return predict_als(playlist_idx, train_matrix, P_als, Q_als)

# Evaluate the ALS model using our generic evaluation function
hit_ratio, mrr, ndcg = evaluate_model(train_matrix, test_items, predict_als_wrapper, k=10)
print("MF-ALS - Hit Ratio @10:", hit_ratio)
print("MF-ALS - MRR @10:", mrr)
print("MF-ALS - NDCG @10:", ndcg)
print("Scores:", hit_ratio + mrr + ndcg)

MF-ALS - Hit Ratio @10: 0.986
MF-ALS - MRR @10: 0.8500323885464124
MF-ALS - NDCG @10: 0.8837706685226564
Scores: 2.7198030570690688
